# Shoof layer model (step 2 of 2)
Trains the small "top layer" model on the data from step 1 and exports it for the browser.

**Before running:** right panel → **Add Input** → *Your Work* → pick the step 1 notebook (its output appears under `/kaggle/input/`).
**Settings:** Internet **On**, Accelerator **GPU T4 x2** (or P100). Then **Save Version → Save & Run All**. About 3–5 hours. Download `shoof-layers-fp16.onnx`, `preview.png` and `report.json` from the Output tab.

In [ ]:
EPOCHS = 12
BATCH = 32
!git clone -q --depth 1 https://github.com/shahrosekhalid96-debug/shoof /kaggle/temp/shoof
!pip install -q onnx onnxconverter-common
import glob, os, tarfile, shutil
print('Inputs attached:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'none')
os.makedirs('/kaggle/temp/ld', exist_ok=True)
# Step 1 output: ld-*.tar files (Kaggle sometimes unpacks them into folders instead)
tars = sorted(glob.glob('/kaggle/input/**/ld-*.tar', recursive=True))
for t in tars:
    tarfile.open(t).extractall('/kaggle/temp/ld')
for m in glob.glob('/kaggle/input/**/meta.json', recursive=True):
    d = os.path.dirname(m); dst = os.path.join('/kaggle/temp/ld', os.path.basename(d))
    if not os.path.exists(dst): os.symlink(d, dst)
n = len(os.listdir('/kaggle/temp/ld'))
print(len(tars), 'tar files,', n, 'designs ready')
assert n, ('No training data found. Check: 1) the step 1 notebook finished (green tick) and its Output tab shows ld-00.tar...; '
           '2) it is added here: right panel -> Add Input -> Your Work -> step 1 notebook.')

In [ ]:
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - turn on the accelerator')
!python /kaggle/temp/shoof/tools/layerdata/train.py --data /kaggle/temp/ld --out /kaggle/working --epochs {EPOCHS} --batch {BATCH} --size 512 --workers 4 --pretrained

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/preview.png'))
print(open('/kaggle/working/report.json').read())